# 1. Load data


In [2]:
import pandas as pd

path = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year7_125activities.csv"

df = pd.read_csv(path)

# 2. Select the three candidate variable groups
MEMS7GR_ALL is the summary column, excluded here


In [3]:
mems7gr_cols = [c for c in df.columns if c.startswith('MEMS7GR_') and c != 'MEMS7GR_ALL']
months12_cols = [c for c in df.columns if c.startswith('MONTHS_12_')]
days_cols = [c for c in df.columns if c.startswith('DAYS10P60GR_')]


# 3. Compute missing rate per activity


In [4]:
def missing_summary(cols, prefix):
    return pd.Series({c.replace(prefix, ''): df[c].isna().mean() for c in cols})

mems7gr_missing = missing_summary(mems7gr_cols, 'MEMS7GR_')
months12_missing = missing_summary(months12_cols, 'MONTHS_12_')
days_missing = missing_summary(days_cols, 'DAYS10P60GR_')


# 4. Combine into one comparison table
Activities with the highest missing rate
Count activities with zero missing vs high missing

In [5]:
missing_table = pd.DataFrame({
    'MEMS7GR': mems7gr_missing,
    'MONTHS_12': months12_missing,
    'DAYS10P60GR': days_missing
})

print('Overall distribution')
print(missing_table.describe())

print(missing_table.sort_values('MEMS7GR', ascending=False).head(10))

print('Activities with zero missing across all three:', (missing_table == 0).all(axis=1).sum())
print('Activities with MEMS7GR missing rate above 20%:', (missing_table['MEMS7GR'] > 0.2).sum())


Overall distribution
          MEMS7GR   MONTHS_12  DAYS10P60GR
count  125.000000  125.000000   125.000000
mean     0.103510    0.103510     0.103510
std      0.119167    0.119167     0.119167
min      0.000000    0.000000     0.000000
25%      0.000000    0.000000     0.000000
50%      0.000000    0.000000     0.000000
75%      0.239606    0.239606     0.239606
max      0.239606    0.239606     0.239606
                     MEMS7GR  MONTHS_12  DAYS10P60GR
MARTIALCHINESE_S05  0.239606   0.239606     0.239606
AIRGUN_S08          0.239606   0.239606     0.239606
MARTIALOTHER_S06    0.239606   0.239606     0.239606
RIFLE_S09           0.239606   0.239606     0.239606
SKIING_T01          0.239606   0.239606     0.239606
RUGBYUNTOUCH_Q14    0.239606   0.239606     0.239606
CLIMBWALL_R02       0.239606   0.239606     0.239606
MOTORCARRACE_U28    0.239606   0.239606     0.239606
MOTORCYCRACE_U27    0.239606   0.239606     0.239606
GYMNASTICSONLY_U24  0.239606   0.239606     0.239606
Activitie

# 5. Reshape into long format, combining three participation candidates

In [6]:
id_cols = ['serial', 'wt_final', 'wt_time', 'Age16plus', 'Age9', 'Disab3'] + \
          [c for c in df.columns if c.startswith('disty')]

activities = [c.replace('MEMS7GR_', '') for c in mems7gr_cols]

frames = []
for act in activities:
    sub = df[id_cols + [f'MEMS7GR_{act}', f'MONTHS_12_{act}', f'DAYS10P60GR_{act}']].copy()
    sub['activity'] = act
    sub = sub.rename(columns={
        f'MEMS7GR_{act}': 'MEMS7GR',
        f'MONTHS_12_{act}': 'MONTHS_12',
        f'DAYS10P60GR_{act}': 'DAYS10P60GR'
    })
    frames.append(sub)

long_df = pd.concat(frames, ignore_index=True)

print(long_df.shape)
output_path = r"C:\Users\Lenovo\Desktop\Dissertation\DATa\7.12-15_RQ3_data\year7_long_table.csv"
long_df.to_csv(output_path, index=False)


(2017375, 23)
